# syft-ingest API Exploration

In [1]:
import sys
from pathlib import Path

# Add parent directory to Python path so imports work
notebook_dir = Path(".").resolve()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

print(f"✅ Added to sys.path: {notebook_dir}")

✅ Added to sys.path: /Users/khoaguin/Desktop/projects/OpenMined/syfthub-stack/syft-ingest/.trees/data-acquisition-2/notebooks


## 1. Environment check

In [2]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)

BRIGHTDATA_API_TOKEN : 7a5...1d5


## 2. Core imports

In [ ]:
from syft_ingest.core.fetcher import (
    FetchAuthError,
    FetchConfig,
    FetchEmptyResultError,
    FetchError,
    FetchRequest,
)
from syft_ingest.core.registry import FETCHER_REGISTRY, get_fetcher
from syft_ingest.core.url_router import Platform

print("Imports OK")

## 3. Registry state

Importing a fetcher module auto-registers it. Check what's registered.

In [4]:
try:
    import syft_ingest.sources.brightdata  # registers facebook + instagram
except Exception as e:
    print(f"BrightData import failed (token missing?): {e}")

import syft_ingest.sources.youtube  # registers youtube

print("Registered fetchers:")
if not FETCHER_REGISTRY:
    print("  (none)")
for key, fetcher in FETCHER_REGISTRY.items():
    print(f"  {key.platform.value}/{key.extractor} -> {type(fetcher).__name__}")

BrightData import failed (token missing?): No module named 'syft_ingest.sources.brightdata'


ModuleNotFoundError: No module named 'syft_ingest.sources.youtube'

## 4. YtDlpFetcher — single video

In [ ]:
yt = YtDlpFetcher()

video_request = FetchRequest(
    platform="youtube",
    urls=["https://www.youtube.com/watch?v=kCc8FmEb1nY"],  # Karpathy: GPT from scratch
    config=FetchConfig(socket_timeout=60),  # Validated config with IDE support
)

video_result = yt.fetch(video_request)
print(f"rows_fetched : {video_result.rows_fetched}")
print(f"remote_job_id: {video_result.remote_job_id}")

In [ ]:
v = video_result.items[0]
print(type(v).__name__)
print(v.model_dump(exclude_none=True))

## 5. YtDlpFetcher — channel enumeration

In [ ]:
channel_request = FetchRequest(
    platform="youtube",
    urls=["https://www.youtube.com/@AndrejKarpathy"],
    config=FetchConfig(playlistend=5),  # Cap at 5 to keep it fast
)

channel_result = yt.fetch(channel_request)
print(f"Videos found: {channel_result.rows_fetched}")
for item in channel_result.items:
    print(f"  [{item.published_at}] {item.title}")

## 6. BrightDataFetcher — Facebook

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [ ]:
from syft_ingest.sources.brightdata import BrightDataFetcher

bd = BrightDataFetcher()  # reads BRIGHTDATA_API_TOKEN from env

fb_request = FetchRequest(
    platform="facebook",
    urls=["https://www.facebook.com/karpathy"],
    config=FetchConfig(timeout=120, poll_interval=5),
)

fb_result = bd.fetch(fb_request)
print(f"rows_fetched : {fb_result.rows_fetched}")
print(f"remote_job_id: {fb_result.remote_job_id}")
print(f"fetched_at   : {fb_result.fetched_at}")

In [ ]:
for item in fb_result.items[:3]:
    print(type(item).__name__)
    print(f"  title : {item.title}")
    print(f"  text  : {(item.text or '')[:120]}")
    print(f"  url   : {item.url}")
    print()

## 7. BrightDataFetcher — Instagram

In [ ]:
ig_request = FetchRequest(
    platform="instagram",
    urls=["https://www.instagram.com/karpathy/"],
    config=FetchConfig(
        timeout=120, poll_interval=5, posts_limit=5
    ),  # Limit posts for testing
)

ig_result = bd.fetch(ig_request)
print(f"rows_fetched: {ig_result.rows_fetched}")
for item in ig_result.items[:3]:
    print(type(item).__name__, "|", item.title)

## 8. Error handling

In [ ]:
# Nonexistent video -> expect FetchEmptyResultError or FetchError
try:
    bad = FetchRequest(
        platform=Platform.YOUTUBE,
        extractor="yt-dlp",
        urls=["https://www.youtube.com/watch?v=DOESNOTEXIST000"],
    )
    yt.fetch(bad)
except FetchEmptyResultError as e:
    print(f"FetchEmptyResultError: {e.message}")
except FetchError as e:
    print(f"FetchError ({type(e).__name__}): {e.message}")

In [ ]:
# Bad token -> expect FetchAuthError
try:
    bad_bd = BrightDataFetcher(token="invalid-token")
    bad_bd.fetch(fb_request)
except FetchAuthError as e:
    print(f"FetchAuthError: {e.message}")
except FetchError as e:
    print(f"FetchError ({type(e).__name__}): {e.message}")

## 9. Registry dispatch

After Phase 4 wiring, callers go through the registry rather than instantiating fetchers directly.

In [ ]:
fetcher = get_fetcher(Platform.YOUTUBE, "yt-dlp")
print(type(fetcher).__name__)

result = fetcher.fetch(
    FetchRequest(
        platform="youtube",
        urls=["https://www.youtube.com/watch?v=kCc8FmEb1nY"],
        config=FetchConfig(),  # Validated config model
    )
)
print(f"Got {result.rows_fetched} item(s) via registry dispatch")

In [ ]:
# Content hashes -- foundation for Phase 6 delta tracking
print("Content hashes:")
for key, sha in result.content_hashes.items():
    print(f"  {str(key)[:60]} -> {sha[:16]}...")

In [ ]:
# FetchConfig provides validation and IDE autocomplete
# All these options are validated:

config = FetchConfig(
    # YouTube options
    socket_timeout=60,  # Network timeout (seconds)
    playlistend=10,  # Max videos from channel
    download_full_video=False,  # Enable video download
    # BrightData options
    timeout=300,  # Scrape job timeout
    poll_interval=2,  # Check frequency
    posts_limit=5,  # Limit posts (testing)
)

print("FetchConfig created with validation:")
print(f"  socket_timeout: {config.socket_timeout}")
print(f"  posts_limit: {config.posts_limit}")
print()

# Invalid values are caught immediately
try:
    bad = FetchConfig(socket_timeout=-1)  # Must be >= 1
except Exception as e:
    print(f"Validation error (expected): {type(e).__name__}")

In [ ]:
import os
import sys

# Verify we're in the right directory
cwd = os.getcwd()
print(f"Working directory: {cwd}")

# Check if syft_ingest can be imported
try:
    import syft_ingest

    print(f"✅ syft_ingest imported from: {syft_ingest.__file__}")
except ImportError as e:
    print(f"❌ Failed to import syft_ingest: {e}")

# Check BRIGHTDATA_API_TOKEN
token = os.getenv("BRIGHTDATA_API_TOKEN")
print(f"BRIGHTDATA_API_TOKEN: {'SET' if token else 'NOT SET'}")